# two-plates-peak-fit.ipynb
Fits the g_z(O) peaks between the plates and shows the peak-to-peak distance and each peak's distance from its plate.

- **Runs on:** PC
- **Inputs:** `<d>.out` in `GZ_DATA_DIR`; one-plate `gZ-one-plate-300K-averaged-left/right.dat` in `one-plate/results_pc/`
- **Outputs:** `gZ/results_pc/two-plates-peak-fit/<d>_peak_distance.png`
- Run **Setup** first.

## Setup
Paths and `load_gz(d)`. **Change:** `GZ_DATA_DIR`.

In [ ]:
from pathlib import Path
import pandas as pd

OUT = Path.cwd() / "results_pc"                      # outputs (runs on PC)
GZ_DATA_DIR = Path("/gibbs/arghavan/TODO_gZ_data")    # two-plate <d>.out files (d = distance x 10); set after PC listing
ONE_PLATE_DIR = Path.cwd().parent / "one-plate" / "results_pc"   # one-plate averaged left/right g(z)
N_VOXELS = 170 * 170                                 # y-z voxels per x slab


def load_gz(d):
    """Two-plate g_z(O) for distance d (x10, e.g. 92): reads <d>.out, divides counts by N_VOXELS."""
    data = pd.read_csv(GZ_DATA_DIR / f"{d}.out", sep=r"\s+", header=None)
    return data[0], data[1] / N_VOXELS

## Test: one distance
Run first to check the plot; saving is commented out. **Change:** `d` in `for d in [92]`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import matplotlib.ticker as ticker 
import pandas as pd
import os

leftside_density_file = ONE_PLATE_DIR / 'gZ-one-plate-300K-averaged-left.dat'
rightside_density_file = ONE_PLATE_DIR / 'gZ-one-plate-300K-averaged-right.dat'

leftside_df = pd.read_csv(leftside_density_file, header=0, sep='\s+')
rightside_df = pd.read_csv(rightside_density_file, header=0, sep='\s+')

x_left = leftside_df['x(Angstrom)']
numDens_left = leftside_df['gO(z)_total']

x_right = rightside_df['x(Angstrom)']
numDens_right = rightside_df['gO(z)_total']

output_path = OUT / 'two-plates-peak-fit'
os.makedirs(output_path, exist_ok=True)

for d in [92]:   # test: one distance
    d_in_angstrom = d / 10
    plt.figure(figsize=(12, 6))

    # ===========================================================================================================================
    #                                                       Loading Data
    # ===========================================================================================================================

    twoplate_x, twoplate_numDens = load_gz(d)

    # ------

    twoplate_x_centered = (twoplate_x - 25).round(2)

    # ===========================================================================================================================
    #                       shift the one-plate graphs to align with the start and end point of two-plate graph
    # ===========================================================================================================================

    x_left_shifted = (twoplate_x_centered.iloc[0] + (x_left - x_left.iloc[0])).round(2)
    x_right_shifted = (twoplate_x_centered.iloc[-1] - (x_right.iloc[-1] - x_right)).round(2)

    # ===========================================================================================================================
    #          truncate the one-plate datasets to only include the region where they overlap with the two-plate dataset
    # ===========================================================================================================================

    x_min = np.min(twoplate_x_centered)
    x_max = np.max(twoplate_x_centered)


    # Truncate left dataset
    mask_left = (x_left_shifted >= x_min) & (x_left_shifted <= x_max)
    x_left_trimmed = x_left_shifted[mask_left]
    numDens_left_trim = numDens_left[mask_left]

    # Truncate right dataset
    mask2 = (x_right_shifted >= x_min) & (x_right_shifted <= x_max)
    x_right_trimmed = x_right_shifted[mask2]
    numDens_right_trim = numDens_right[mask2]
    # ===========================================================================================================================
    #                                         Interference of the two one-plate graphs
    # ===========================================================================================================================

    interference = numDens_left_trim.values + numDens_right_trim.values - 1
    interference = np.clip(interference, a_min=0, a_max=None)

    twoplate_numDens_vals = twoplate_numDens.values

    # ===========================================================================================================================
    #                 Truncatung data to only include the region where densities between the plates is non-zero
    # ===========================================================================================================================


    first_nonzero = np.argmax(twoplate_numDens_vals > 0.05)  # first index where twoplate_numDens != 0
    last_nonzero = len(twoplate_numDens_vals) - np.argmax(twoplate_numDens_vals[::-1] > 0.05) - 1  # last index where twoplate_numDens != 0

    n_keep_zeros = 0 # number of zeros to keep before the first non-zero

    # compute starting index safely
    start_idx = max(0, first_nonzero - n_keep_zeros)
    end_idx = min(len(twoplate_numDens_vals), last_nonzero + n_keep_zeros + 1) 


    mask = np.zeros_like(twoplate_numDens_vals, dtype=bool)
    mask[start_idx:end_idx] = True  # only keep this region

    x_left_trimmed_masked = x_left_trimmed[mask]
    numDens_left_trim_masked = numDens_left_trim[mask]

    x_right_trimmed_masked = x_right_trimmed[mask]
    numDens_right_trim_masked = numDens_right_trim[mask]

    interference_masked = interference[mask]

    # ===========================================================================================================================
    #                                                   fitting peaks
    # ===========================================================================================================================


    peak_indices, _ = find_peaks(twoplate_numDens, height=1.2)                    # any maximum lower than 1.2 is not considered a peak (to avoid minimums and only look for peaks above bulk water level)
    x_peaks_init = twoplate_x_centered[peak_indices]   


    fitted_peaks = []


    degree = 4

    for x0_guess in x_peaks_init:
        
        # mask ±0.25 around initial peak guess
        mask = (twoplate_x_centered > x0_guess - 0.25) & (twoplate_x_centered < x0_guess + 0.25)
        x_fit = twoplate_x_centered[mask]
        y_fit = twoplate_numDens[mask]
        
        if len(x_fit) < degree + 1:
            continue
        coeffs = np.polyfit(x_fit, y_fit, deg=degree)
        poly = np.poly1d(coeffs)

        x_dense = np.linspace(x_fit.min(), x_fit.max(), 500)
        y_dense = poly(x_dense)
        max_idx = np.argmax(y_dense)
        x_peak = x_dense[max_idx]
        y_peak = y_dense[max_idx] 

        fitted_peaks.append((x_peak, y_peak))
        plt.plot(x_dense, y_dense, 'tab:cyan', linewidth = 0.7, zorder=3)
        
    # show the distance 
    if len(fitted_peaks) >= 2:
        fitted_peaks.sort()  # sort by x
        x1, y1 = fitted_peaks[0]
        x2, y2 = fitted_peaks[1]
        y_line = max(y1, y2)
        peak1_position = (d_in_angstrom/2) + x1
        peak2_position = d_in_angstrom - ((d_in_angstrom/2) + x2)
        # plt.plot(x1, y_line+0.1, 'rx', label=f'Peak distance from the left plate ≈  {peak1_position:.2f} Å', alpha=0.7)
        # plt.plot(x2, y_line+0.1, 'rx', label=f'Peak distance from the right plate ≈  {peak2_position:.2f} Å', alpha=0.7)
        plt.plot(x1, y_line+0.1, color='gray' , marker = 'o', markersize=3, label=f'Peak distance from the left plate ≈  {peak1_position:.2f} Å', alpha=0.7)
        plt.plot(x2, y_line+0.1, color='gray' , marker = 'o', markersize=3, label=f'Peak distance from the right plate ≈  {peak2_position:.2f} Å', alpha=0.7)
        plt.hlines(y=y_line+0.1, xmin=x1, xmax=x2, color='gray', linestyle='--')
        plt.text((x1 + x2) / 2, y_line + 0.2, f"{x2 - x1:.2f} Å", ha='center', fontsize=8, color='gray')



    # ===========================================================================================================================
    #                                                   Plotting
    # ===========================================================================================================================


    plt.plot(twoplate_x_centered, twoplate_numDens, 'bo-', markersize=3, linewidth=1, label=r'$g_z(O)$ between the Plates')
    # plt.plot(x_left_trimmed_masked, numDens_left_trim_masked, label='Left Plate', marker='o', color='green',markersize='1.5', alpha=0.3)
    # plt.plot(x_right_trimmed_masked, numDens_right_trim_masked, label='Right Plate', marker='o', color='orange',markersize='1.5', alpha=0.3)
    plt.plot(x_left_trimmed_masked, interference_masked, label='RDF Interference', linestyle='-', linewidth=1, color='red', alpha=0.8)


    plt.hlines(y=1.00, xmin=-8, xmax=8, color='black', linestyle='--', linewidth=1, label='Bulk Water Density')

    plt.text(-6, 5.5, f"Plate Distance: {d_in_angstrom} Å", ha='center', fontsize=11, color='dimgrey',
                bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.3'))

    plt.title(f'Number Density Between the Plates')
    plt.xlabel('Distance from center (Å)')
    plt.ylabel(r'$g_z(O)$')
    plt.legend(fontsize='x-small')
    plt.ylim(-0.2, 6)
    plt.xlim(-8,8)

    plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(1))
    plt.gca().xaxis.set_minor_locator(ticker.MultipleLocator(0.1))
    
    # plt.savefig(f'{output_path}/{d_in_angstrom}_peak_distance.png', dpi=600)

## All distances, 7.6–10.2 Å
4th-order polynomial fit (±0.25 Å) around each peak above 1.2; saves one plot per distance. **Change:** distance range, `height`, `degree`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import matplotlib.ticker as ticker 
import pandas as pd
import os

leftside_density_file = ONE_PLATE_DIR / 'gZ-one-plate-300K-averaged-left.dat'
rightside_density_file = ONE_PLATE_DIR / 'gZ-one-plate-300K-averaged-right.dat'

leftside_df = pd.read_csv(leftside_density_file, header=0, sep='\s+')
rightside_df = pd.read_csv(rightside_density_file, header=0, sep='\s+')

x_left = leftside_df['x(Angstrom)']
numDens_left = leftside_df['gO(z)_total']

x_right = rightside_df['x(Angstrom)']
numDens_right = rightside_df['gO(z)_total']

output_path = OUT / 'two-plates-peak-fit'
os.makedirs(output_path, exist_ok=True)

for d in range(76, 103, 2):
# d  = 92
    d_in_angstrom = d / 10
    plt.figure(figsize=(12, 6))

    # ===========================================================================================================================
    #                                                       Loading Data
    # ===========================================================================================================================

    twoplate_x, twoplate_numDens = load_gz(d)

    # ------

    twoplate_x_centered = (twoplate_x - 25).round(2)

    # ===========================================================================================================================
    #                       shift the one-plate graphs to align with the start and end point of two-plate graph
    # ===========================================================================================================================

    x_left_shifted = (twoplate_x_centered.iloc[0] + (x_left - x_left.iloc[0])).round(2)
    x_right_shifted = (twoplate_x_centered.iloc[-1] - (x_right.iloc[-1] - x_right)).round(2)

    # ===========================================================================================================================
    #          truncate the one-plate datasets to only include the region where they overlap with the two-plate dataset
    # ===========================================================================================================================

    x_min = np.min(twoplate_x_centered)
    x_max = np.max(twoplate_x_centered)


    # Truncate left dataset
    mask_left = (x_left_shifted >= x_min) & (x_left_shifted <= x_max)
    x_left_trimmed = x_left_shifted[mask_left]
    numDens_left_trim = numDens_left[mask_left]

    # Truncate right dataset
    mask2 = (x_right_shifted >= x_min) & (x_right_shifted <= x_max)
    x_right_trimmed = x_right_shifted[mask2]
    numDens_right_trim = numDens_right[mask2]
    # ===========================================================================================================================
    #                                         Interference of the two one-plate graphs
    # ===========================================================================================================================

    interference = numDens_left_trim.values + numDens_right_trim.values - 1
    interference = np.clip(interference, a_min=0, a_max=None)

    twoplate_numDens_vals = twoplate_numDens.values

    # ===========================================================================================================================
    #                 Truncatung data to only include the region where densities between the plates is non-zero
    # ===========================================================================================================================


    first_nonzero = np.argmax(twoplate_numDens_vals > 0.05)  # first index where twoplate_numDens != 0
    last_nonzero = len(twoplate_numDens_vals) - np.argmax(twoplate_numDens_vals[::-1] > 0.05) - 1  # last index where twoplate_numDens != 0

    n_keep_zeros = 0 # number of zeros to keep before the first non-zero

    # compute starting index safely
    start_idx = max(0, first_nonzero - n_keep_zeros)
    end_idx = min(len(twoplate_numDens_vals), last_nonzero + n_keep_zeros + 1) 


    mask = np.zeros_like(twoplate_numDens_vals, dtype=bool)
    mask[start_idx:end_idx] = True  # only keep this region

    x_left_trimmed_masked = x_left_trimmed[mask]
    numDens_left_trim_masked = numDens_left_trim[mask]

    x_right_trimmed_masked = x_right_trimmed[mask]
    numDens_right_trim_masked = numDens_right_trim[mask]

    interference_masked = interference[mask]

    # ===========================================================================================================================
    #                                                   fitting peaks
    # ===========================================================================================================================


    peak_indices, _ = find_peaks(twoplate_numDens, height=1.2)                    # any maximum lower than 1.2 is not considered a peak (to avoid minimums and only look for peaks above bulk water level)
    x_peaks_init = twoplate_x_centered[peak_indices]   


    fitted_peaks = []


    degree = 4

    for x0_guess in x_peaks_init:
        
        # mask ±0.25 around initial peak guess
        mask = (twoplate_x_centered > x0_guess - 0.25) & (twoplate_x_centered < x0_guess + 0.25)
        x_fit = twoplate_x_centered[mask]
        y_fit = twoplate_numDens[mask]
        
        if len(x_fit) < degree + 1:
            continue
        coeffs = np.polyfit(x_fit, y_fit, deg=degree)
        poly = np.poly1d(coeffs)

        x_dense = np.linspace(x_fit.min(), x_fit.max(), 500)
        y_dense = poly(x_dense)
        max_idx = np.argmax(y_dense)
        x_peak = x_dense[max_idx]
        y_peak = y_dense[max_idx] 

        fitted_peaks.append((x_peak, y_peak))
        plt.plot(x_dense, y_dense, 'tab:cyan', linewidth = 0.7, zorder=3)
        
    # show the distance 
    if len(fitted_peaks) >= 2:
        fitted_peaks.sort()  # sort by x
        x1, y1 = fitted_peaks[0]
        x2, y2 = fitted_peaks[1]
        y_line = max(y1, y2)
        peak1_position = (d_in_angstrom/2) + x1
        peak2_position = d_in_angstrom - ((d_in_angstrom/2) + x2)
        # plt.plot(x1, y_line+0.1, 'rx', label=f'Peak distance from the left plate ≈  {peak1_position:.2f} Å', alpha=0.7)
        # plt.plot(x2, y_line+0.1, 'rx', label=f'Peak distance from the right plate ≈  {peak2_position:.2f} Å', alpha=0.7)
        plt.plot(x1, y_line+0.1, color='gray' , marker = 'o', markersize=3, label=f'Peak distance from the left plate ≈  {peak1_position:.2f} Å', alpha=0.7)
        plt.plot(x2, y_line+0.1, color='gray' , marker = 'o', markersize=3, label=f'Peak distance from the right plate ≈  {peak2_position:.2f} Å', alpha=0.7)
        plt.hlines(y=y_line+0.1, xmin=x1, xmax=x2, color='gray', linestyle='--')
        plt.text((x1 + x2) / 2, y_line + 0.2, f"{x2 - x1:.2f} Å", ha='center', fontsize=8, color='gray')



    # ===========================================================================================================================
    #                                                   Plotting
    # ===========================================================================================================================


    plt.plot(twoplate_x_centered, twoplate_numDens, 'bo-', markersize=3, linewidth=1, label=r'$g_z(O)$ between the Plates')
    # plt.plot(x_left_trimmed_masked, numDens_left_trim_masked, label='Left Plate', marker='o', color='green',markersize='1.5', alpha=0.3)
    # plt.plot(x_right_trimmed_masked, numDens_right_trim_masked, label='Right Plate', marker='o', color='orange',markersize='1.5', alpha=0.3)
    plt.plot(x_left_trimmed_masked, interference_masked, label='RDF Interference', linestyle='-', linewidth=1, color='red', alpha=0.8)


    plt.hlines(y=1.00, xmin=-8, xmax=8, color='black', linestyle='--', linewidth=1, label='Bulk Water Density')

    plt.text(-6, 5.5, f"Plate Distance: {d_in_angstrom} Å", ha='center', fontsize=11, color='dimgrey',
                bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.3'))

    plt.title(f'Number Density Between the Plates')
    plt.xlabel('Distance from center (Å)')
    plt.ylabel(r'$g_z(O)$')
    plt.legend(fontsize='x-small')
    plt.ylim(-0.2, 6)
    plt.xlim(-8,8)

    plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(1))
    plt.gca().xaxis.set_minor_locator(ticker.MultipleLocator(0.1))
    
    plt.savefig(f'{output_path}/{d_in_angstrom}_peak_distance.png', dpi=600)